# Embryo AI Model — Diagnostic Notebook

**Goal:** Identify why the model assigns moderately high euploid scores to arrested embryos that show little or no cell division by Day 5.

**Run each step in order.** Each step builds on the previous one and ends with a clear finding that guides the next step.

| Step | What we check |
|------|---------------|
| 1 | Setup & paths |
| 2 | Load the model |
| 3 | Explore annotation data |
| 4 | Basic inference sanity check |
| 5 | Score distribution on the full gold test set |
| 6 | Arrested vs normal embryo comparison |
| 7 | Grad-CAM — what does the model actually look at? |
| 8 | Heuristic pre-filter evaluation |
| 9 | Summary & decision matrix |

In [1]:
# Auto-install missing packages into whichever Python kernel is running.
# Safe to re-run — skips already-installed packages.
import subprocess, sys

required = [
    'numpy', 'pandas', 'matplotlib', 'torch', 'torchvision',
    'scipy', 'opencv-python', 'pillow', 'tqdm', 'scikit-learn',
]

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + required,
    capture_output=True, text=True
)
if result.returncode != 0:
    print('pip stderr:', result.stderr[-500:])
else:
    print(f'All packages ready  (Python: {sys.executable})')

All packages ready  (Python: /opt/homebrew/opt/python@3.11/bin/python3.11)


---
## Step 1 — Setup
Set paths and import everything the notebook needs.

In [ ]:
!pip install numpy

In [2]:
import sys, os
from pathlib import Path

# ── Notebook is expected to live in embryo-ai-backend/ ────────────────────────
REPO = Path(".").resolve()
SRC  = REPO / "src"
sys.path.insert(0, str(SRC))

DATA_DIR   = REPO / "data" / "blastocyst"
IMAGES_DIR = DATA_DIR / "Images"
MODELS_DIR = REPO / "models"

print("REPO     :", REPO)
print("Images   :", IMAGES_DIR, "| exists:", IMAGES_DIR.exists())
print("Weights  :", MODELS_DIR / "blastocyst_grader.pt",
      "| exists:", (MODELS_DIR / "blastocyst_grader.pt").exists())

# ── Core imports ──────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # works in headless / VSCode notebooks
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn.functional as F
from PIL import Image
import cv2
import warnings
warnings.filterwarnings("ignore")

OUTPUT_DIR = REPO / "diag_output"
OUTPUT_DIR.mkdir(exist_ok=True)
print("\nAll imports OK. Outputs will be saved to:", OUTPUT_DIR)

REPO     : /Users/sudaynandansamala/Documents/projects/IVF/embryo-ai-backend
Images   : /Users/sudaynandansamala/Documents/projects/IVF/embryo-ai-backend/data/blastocyst/Images | exists: True
Weights  : /Users/sudaynandansamala/Documents/projects/IVF/embryo-ai-backend/models/blastocyst_grader.pt | exists: True

All imports OK. Outputs will be saved to: /Users/sudaynandansamala/Documents/projects/IVF/embryo-ai-backend/diag_output


---
## Step 2 — Load the Model
We import the `BlastocystGrader` class directly from `src/model.py` so we can access internals (heads, backbone layers) for Grad-CAM later.

**Expected output:** Model loads without error, mode = `REAL inference`, not mock.

In [3]:
from model import (
    BlastocystGrader,
    INFERENCE_TRANSFORM,
    WEIGHTS_PATH,
    DEVICE,
    IDX_TO_BE,
    IDX_TO_ICM,
    IDX_TO_TE,
)

# ── Build model and load weights ──────────────────────────────────────────────
model = BlastocystGrader(pretrained=False).to(DEVICE)

if not WEIGHTS_PATH.exists():
    raise FileNotFoundError(f"No weights at {WEIGHTS_PATH}. Run scripts/train.py first.")

state = torch.load(WEIGHTS_PATH, map_location=DEVICE)
model.load_state_dict(state)
model.eval()

print(f"Model loaded successfully")
print(f"  Device  : {DEVICE}")
print(f"  Weights : {WEIGHTS_PATH.name}  ({WEIGHTS_PATH.stat().st_size / 1e6:.1f} MB)")
print(f"  Heads   : BE={model.be_head[-1].out_features} classes, "
      f"ICM={model.icm_head[-1].out_features} classes, "
      f"TE={model.te_head[-1].out_features} classes")

# ── Convenience inference function ────────────────────────────────────────────
@torch.no_grad()
def infer(img_path: str) -> dict:
    """Run one image through the model. Returns raw probabilities and predicted classes."""
    img = Image.open(img_path).convert("RGB")
    x   = INFERENCE_TRANSFORM(img).unsqueeze(0).to(DEVICE)
    out = model(x)
    be_probs  = F.softmax(out["BE"],  dim=1).squeeze(0).cpu().numpy()
    icm_probs = F.softmax(out["ICM"], dim=1).squeeze(0).cpu().numpy()
    te_probs  = F.softmax(out["TE"],  dim=1).squeeze(0).cpu().numpy()

    be_score  = float(np.dot(be_probs,  [i / (len(be_probs) - 1)  for i in range(len(be_probs))]))
    icm_score = float(np.dot(icm_probs, [i / (len(icm_probs) - 1) for i in range(len(icm_probs))]))
    te_score  = float(np.dot(te_probs,  [i / (len(te_probs) - 1)  for i in range(len(te_probs))]))

    expansion    = be_score * 5.0 + 1.0
    blast_score  = expansion * 2.0 + icm_score * 4.0 + te_score * 4.0
    base_ploidy  = be_score * 0.4 + icm_score * 0.3 + te_score * 0.3
    euploid_prob = float(np.clip(base_ploidy, 0.02, 0.98))

    return {
        "be_probs":    be_probs,
        "icm_probs":   icm_probs,
        "te_probs":    te_probs,
        "be_score":    be_score,
        "icm_score":   icm_score,
        "te_score":    te_score,
        "expansion":   expansion,
        "blast_score": blast_score,
        "euploid_prob": euploid_prob,
        "predicted":   euploid_prob >= 0.5,
    }

print("\ninfer() helper ready.")

Model loaded successfully
  Device  : mps
  Weights : blastocyst_grader.pt  (59.1 MB)
  Heads   : BE=5 classes, ICM=4 classes, TE=4 classes

infer() helper ready.


---
## Step 3 — Explore Annotation Data

We use the gold-standard test set (`Gardner_test_gold_onlyGardnerScores.csv`): 300 images, labelled by expert annotators.

**Label encoding (EXP_gold):**
- `0` = arrested / no blastulation (degenerate)
- `1` = early blastocyst (cavitation starting)
- `2` = full blastocyst
- `3` = expanded blastocyst
- `4` = hatching / hatched

ICM / TE: `0`=C (poor) · `1`=B (fair) · `2`=A (good) · `ND`/`NA` = not determinable (usually arrested)

In [4]:
# ── Load gold test annotations ────────────────────────────────────────────────
gold = pd.read_csv(DATA_DIR / "Gardner_test_gold_onlyGardnerScores.csv", sep=";")
gold.columns = [c.strip().lstrip("\ufeff") for c in gold.columns]
gold = gold.dropna(subset=["Image", "EXP_gold"]).copy()
gold["EXP_gold"] = gold["EXP_gold"].astype(float).astype(int)

# Keep only images that actually exist on disk
gold["path"] = gold["Image"].apply(lambda n: str(IMAGES_DIR / n.strip()))
gold = gold[gold["path"].apply(os.path.exists)].reset_index(drop=True)

print(f"Gold test samples with images on disk: {len(gold)}")
print()
print("EXP_gold distribution:")
label_names = {0: "Arrested", 1: "Early blast.", 2: "Full blast.",
               3: "Expanded", 4: "Hatching"}
counts = gold["EXP_gold"].value_counts().sort_index()
for grade, cnt in counts.items():
    bar = "█" * cnt
    print(f"  EXP={grade} ({label_names.get(grade,'?'):14s}) | {cnt:3d} | {bar}")

# ── Plot distribution ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3.5))
colors = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#3498db"]
bars = ax.bar([label_names[g] for g in counts.index], counts.values, color=colors)
ax.set_title("Gold Test Set — Expansion Grade Distribution", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of embryos")
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.5, str(v), ha="center", fontsize=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "step3_label_distribution.png", dpi=120)
plt.show()
print("\nSaved → diag_output/step3_label_distribution.png")

Gold test samples with images on disk: 298

EXP_gold distribution:
  EXP=0 (Arrested      ) |  23 | ███████████████████████
  EXP=1 (Early blast.  ) |  31 | ███████████████████████████████
  EXP=2 (Full blast.   ) |  86 | ██████████████████████████████████████████████████████████████████████████████████████
  EXP=3 (Expanded      ) | 153 | █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  EXP=4 (Hatching      ) |   5 | █████

Saved → diag_output/step3_label_distribution.png


---
## Step 4 — Basic Inference Sanity Check

Before running the full test set, verify the model produces sensible outputs on a handful of manually chosen images:
- **2 good embryos** (EXP=3, expanded blastocyst)
- **2 arrested embryos** (EXP=0)

**Expected:** Good embryos → euploid_prob > 0.5, arrested → euploid_prob close to 0.

**Failure signal:** Arrested embryo scores ≥ 0.5 → model is misclassifying them.

In [5]:
arrested_rows = gold[gold["EXP_gold"] == 0].head(2)
good_rows     = gold[gold["EXP_gold"] == 3].head(2)
spot_check    = pd.concat([good_rows, arrested_rows]).reset_index(drop=True)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle("Step 4 — Sanity Check: 2 Good vs 2 Arrested", fontsize=13, fontweight="bold")

for i, row in spot_check.iterrows():
    true_label = label_names[row["EXP_gold"]]
    result     = infer(row["path"])

    # ── Image panel (top row) ─────────────────────────────────────────────────
    img = Image.open(row["path"]).convert("RGB")
    ax  = axes[0, i]
    ax.imshow(img, cmap="gray")
    ax.set_title(f"{row['Image']}\nTrue: {true_label}", fontsize=8)
    ax.axis("off")
    border_color = "#2ecc71" if row["EXP_gold"] >= 2 else "#e74c3c"
    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)

    # ── Score panel (bottom row) ──────────────────────────────────────────────
    ax2 = axes[1, i]
    score_names = ["BE score", "ICM score", "TE score", "Euploid prob"]
    score_vals  = [result["be_score"], result["icm_score"],
                   result["te_score"], result["euploid_prob"]]
    bar_colors  = ["#3498db", "#9b59b6", "#e67e22",
                   "#2ecc71" if result["predicted"] else "#e74c3c"]
    bars = ax2.barh(score_names, score_vals, color=bar_colors, height=0.5)
    ax2.set_xlim(0, 1)
    ax2.axvline(0.5, color="gray", linestyle="--", linewidth=1)
    ax2.set_title(
        f"Euploid: {'✓ PREDICTED' if result['predicted'] else '✗ NOT predicted'}\n"
        f"(prob={result['euploid_prob']:.2f})",
        fontsize=8,
        color="#2ecc71" if result["predicted"] else "#e74c3c"
    )
    for bar, val in zip(bars, score_vals):
        ax2.text(min(val + 0.02, 0.95), bar.get_y() + bar.get_height()/2,
                 f"{val:.2f}", va="center", fontsize=8)

    print(f"  {row['Image']:20s} | True: {true_label:14s} | "
          f"Euploid={result['euploid_prob']:.3f} | "
          f"Predicted={'YES' if result['predicted'] else 'NO '}")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "step4_sanity_check.png", dpi=120)
plt.show()
print("\nSaved → diag_output/step4_sanity_check.png")

  0005_08.png          | True: Expanded       | Euploid=0.426 | Predicted=NO 
  0011_03.png          | True: Expanded       | Euploid=0.393 | Predicted=NO 
  0011_02.png          | True: Arrested       | Euploid=0.505 | Predicted=YES
  0014_05.png          | True: Arrested       | Euploid=0.529 | Predicted=YES

Saved → diag_output/step4_sanity_check.png


---
## Step 5 — Score Distribution on the Full Gold Test Set

Run the model on all available gold test images and record the euploid probability for each. This gives us the full picture before we separate by expansion grade.

**This cell takes 1–3 minutes depending on CPU speed.**

In [6]:
from tqdm.auto import tqdm

results_list = []
for _, row in tqdm(gold.iterrows(), total=len(gold), desc="Inferring"):
    try:
        r = infer(row["path"])
        results_list.append({
            "image":        row["Image"],
            "exp_gold":     row["EXP_gold"],
            "be_score":     r["be_score"],
            "icm_score":    r["icm_score"],
            "te_score":     r["te_score"],
            "expansion":    r["expansion"],
            "blast_score":  r["blast_score"],
            "euploid_prob": r["euploid_prob"],
            "predicted":    r["predicted"],
            "path":         row["path"],
        })
    except Exception as e:
        print(f"  SKIP {row['Image']}: {e}")

results = pd.DataFrame(results_list)
print(f"\nInference complete. {len(results)} / {len(gold)} images processed.")

print("\nGlobal euploid_prob statistics:")
print(results["euploid_prob"].describe().round(3).to_string())

print(f"\nImages predicted euploid (prob ≥ 0.5): "
      f"{results['predicted'].sum()} / {len(results)} "
      f"({results['predicted'].mean()*100:.1f}%)")

# Save intermediate results
results.to_csv(OUTPUT_DIR / "step5_all_results.csv", index=False)
print("Results saved → diag_output/step5_all_results.csv")

Inferring:   0%|          | 0/298 [00:00<?, ?it/s]


Inference complete. 298 / 298 images processed.

Global euploid_prob statistics:
count    298.000
mean       0.415
std        0.065
min        0.305
25%        0.362
50%        0.398
75%        0.469
max        0.553

Images predicted euploid (prob ≥ 0.5): 53 / 298 (17.8%)
Results saved → diag_output/step5_all_results.csv


In [7]:
# ── Plot: overall euploid_prob histogram coloured by true expansion grade ─────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Step 5 — Euploid Probability Distribution (Full Test Set)",
             fontsize=13, fontweight="bold")

# Left: overall histogram
axes[0].hist(results["euploid_prob"], bins=30, color="#3498db", edgecolor="white")
axes[0].axvline(0.5, color="#e74c3c", linestyle="--", linewidth=2, label="Decision threshold (0.5)")
axes[0].set_xlabel("Euploid Probability")
axes[0].set_ylabel("Count")
axes[0].set_title("All embryos")
axes[0].legend()

# Right: per-grade box plot
grade_order = [0, 1, 2, 3, 4]
grade_data  = [results[results["exp_gold"] == g]["euploid_prob"].values
               for g in grade_order]
bp = axes[1].boxplot(grade_data, patch_artist=True, notch=False,
                     medianprops=dict(color="black", linewidth=2))
box_colors = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#3498db"]
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
axes[1].set_xticks(range(1, 6))
axes[1].set_xticklabels([f"EXP={g}\n{label_names[g]}" for g in grade_order],
                        fontsize=8)
axes[1].axhline(0.5, color="gray", linestyle="--", linewidth=1.5, label="Threshold")
axes[1].set_ylabel("Euploid Probability")
axes[1].set_title("By expansion grade")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "step5_score_distribution.png", dpi=120)
plt.show()
print("Saved → diag_output/step5_score_distribution.png")

print("\nMedian euploid_prob per expansion grade:")
for g in grade_order:
    subset = results[results["exp_gold"] == g]["euploid_prob"]
    above  = (subset >= 0.5).sum()
    print(f"  EXP={g} ({label_names[g]:14s}) | "
          f"n={len(subset):3d} | median={subset.median():.3f} | "
          f"false positives (arrested scored ≥0.5): "
          f"{above if g <= 1 else 'N/A'}")

Saved → diag_output/step5_score_distribution.png

Median euploid_prob per expansion grade:
  EXP=0 (Arrested      ) | n= 23 | median=0.524 | false positives (arrested scored ≥0.5): 19
  EXP=1 (Early blast.  ) | n= 31 | median=0.515 | false positives (arrested scored ≥0.5): 20
  EXP=2 (Full blast.   ) | n= 86 | median=0.429 | false positives (arrested scored ≥0.5): N/A
  EXP=3 (Expanded      ) | n=153 | median=0.369 | false positives (arrested scored ≥0.5): N/A
  EXP=4 (Hatching      ) | n=  5 | median=0.348 | false positives (arrested scored ≥0.5): N/A


---
## Step 6 — Arrested vs Normal: Deep Comparison

Separate the test set into **arrested** (EXP=0,1 — should score low) and **normal blastocysts** (EXP=2,3,4 — should score high) and compare all three model outputs: BE, ICM, TE, and euploid probability.

**Key question:** Are all three heads equally confused, or is the problem concentrated in one?
- If ICM is wrong → the model is mistaking debris/fragmentation for ICM cells
- If BE is wrong → the model cannot detect absence of a blastocoel cavity
- If all three are wrong equally → fundamental feature extraction failure

In [8]:
arrested = results[results["exp_gold"] <= 1].copy()
normal   = results[results["exp_gold"] >= 2].copy()

print(f"Arrested group  (EXP=0,1): {len(arrested)} images")
print(f"Normal group    (EXP≥2)  : {len(normal)} images")
print()

metrics = ["be_score", "icm_score", "te_score", "euploid_prob"]
metric_labels = ["BE (Expansion)", "ICM Quality", "TE Quality", "Euploid Probability"]

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle("Step 6 — Arrested (red) vs Normal (green) Embryos",
             fontsize=13, fontweight="bold")

for ax, metric, label in zip(axes, metrics, metric_labels):
    arr_vals = arrested[metric].values
    nor_vals = normal[metric].values
    ax.hist(nor_vals,  bins=20, alpha=0.6, color="#2ecc71", label="Normal (EXP≥2)",   density=True)
    ax.hist(arr_vals,  bins=20, alpha=0.6, color="#e74c3c", label="Arrested (EXP≤1)", density=True)
    ax.axvline(0.5, color="gray", linestyle="--", linewidth=1)
    ax.set_title(label, fontsize=10, fontweight="bold")
    ax.set_xlabel("Score")
    ax.set_ylabel("Density" if metric == "be_score" else "")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "step6_arrested_vs_normal.png", dpi=120)
plt.show()
print("Saved → diag_output/step6_arrested_vs_normal.png")

print("\n── Mean scores: Arrested vs Normal ──────────────────────────────")
print(f"{'Metric':<20} {'Arrested mean':>16} {'Normal mean':>14} {'Delta':>8}")
print("-" * 62)
for metric, label in zip(metrics, metric_labels):
    a_mean = arrested[metric].mean()
    n_mean = normal[metric].mean()
    delta  = n_mean - a_mean
    flag   = "⚠️  LOW SEPARATION" if delta < 0.15 else ""
    print(f"{label:<20} {a_mean:>16.3f} {n_mean:>14.3f} {delta:>8.3f}  {flag}")

# False positive rate: arrested embryos scored ≥ 0.5
fp_rate = (arrested["predicted"] == True).mean()
print(f"\n🔴 False positive rate on arrested embryos: {fp_rate*100:.1f}%")
print(f"   ({int(fp_rate*len(arrested))} out of {len(arrested)} arrested embryos scored as euploid)")

Arrested group  (EXP=0,1): 54 images
Normal group    (EXP≥2)  : 244 images

Saved → diag_output/step6_arrested_vs_normal.png

── Mean scores: Arrested vs Normal ──────────────────────────────
Metric                  Arrested mean    Normal mean    Delta
--------------------------------------------------------------
BE (Expansion)                  0.493          0.616    0.123  ⚠️  LOW SEPARATION
ICM Quality                     0.580          0.217   -0.363  ⚠️  LOW SEPARATION
TE Quality                      0.467          0.275   -0.192  ⚠️  LOW SEPARATION
Euploid Probability             0.511          0.394   -0.117  ⚠️  LOW SEPARATION

🔴 False positive rate on arrested embryos: 72.2%
   (39 out of 54 arrested embryos scored as euploid)


In [9]:
# ── Identify the worst failures: arrested embryos with HIGHEST euploid scores ─
worst_failures = (
    arrested
    .sort_values("euploid_prob", ascending=False)
    .head(6)
    .reset_index(drop=True)
)

print("Top 6 worst failures — arrested embryos with highest euploid scores:")
print(worst_failures[["image", "exp_gold", "be_score", "icm_score",
                       "te_score", "euploid_prob"]].to_string(index=False))

# Visual grid of failure cases
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle("Step 6 — Worst Failures: Arrested Embryos Scored as Euploid",
             fontsize=13, fontweight="bold")

for ax, (_, row) in zip(axes.flat, worst_failures.iterrows()):
    img = Image.open(row["path"]).convert("RGB")
    ax.imshow(img, cmap="gray")
    ax.set_title(
        f"{row['image']}\n"
        f"True: {label_names[row['exp_gold']]}\n"
        f"Euploid prob: {row['euploid_prob']:.2f}  ← WRONG",
        fontsize=8, color="#e74c3c"
    )
    ax.axis("off")
    for spine in ax.spines.values():
        spine.set_edgecolor("#e74c3c")
        spine.set_linewidth(3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "step6_worst_failures.png", dpi=120)
plt.show()
print("Saved → diag_output/step6_worst_failures.png")

Top 6 worst failures — arrested embryos with highest euploid scores:
      image  exp_gold  be_score  icm_score  te_score  euploid_prob
 430_04.png         1  0.452547   0.688701  0.549737      0.552550
0036_02.png         0  0.462884   0.665588  0.546021      0.548636
0017_02.png         0  0.455250   0.671958  0.545159      0.547235
 573_01.png         0  0.470812   0.657896  0.533931      0.545873
 200_02.png         0  0.457280   0.673024  0.534187      0.545075
0039_03.png         0  0.462058   0.652530  0.540204      0.542643
Saved → diag_output/step6_worst_failures.png


---
## Step 7 — Grad-CAM: What Does the Model Actually Look At?

We apply Grad-CAM to the **last convolutional layer** of the VGG16 backbone (`backbone[28]`) for the ICM "A" class — the head most likely to cause false positives.

We visualise the heatmap on each of the 6 worst-failure arrested embryos.

**How to interpret the heatmap:**

| Heatmap concentrated on... | Diagnosis |
|---|---|
| Zona pellucida (outer ring) | Artifact leakage — model learned the zona, not the ICM |
| Random background / well edges | No coherent features — likely data contamination |
| Fragmentation debris | Model confuses debris with real cells |
| Correct inner cell mass region | Features are right — problem is threshold calibration |

In [10]:
# ── Grad-CAM implementation ────────────────────────────────────────────────────
class GradCAM:
    """
    Grad-CAM for the VGG16 backbone.
    target_layer should be model.backbone[28] (last conv layer).
    """
    def __init__(self, model: torch.nn.Module, target_layer: torch.nn.Module):
        self.model       = model
        self._grads      = None
        self._acts       = None
        self._fwd_hook   = target_layer.register_forward_hook(self._save_activation)
        self._bwd_hook   = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self._acts = output

    def _save_gradient(self, module, grad_input, grad_output):
        self._grads = grad_output[0]

    def generate(self, img_path: str, target_head: str, target_class: int) -> np.ndarray:
        """
        Returns a [0,1]-normalised CAM array of shape (320,320).
        target_head: 'BE', 'ICM', or 'TE'
        target_class: class index to backprop through
        """
        img    = Image.open(img_path).convert("RGB")
        tensor = INFERENCE_TRANSFORM(img).unsqueeze(0).to(DEVICE)
        tensor.requires_grad_(True)

        self.model.zero_grad()
        out  = self.model(tensor)
        prob = F.softmax(out[target_head], dim=1)[0, target_class]
        prob.backward()

        grads = self._grads.detach()          # (1, C, H, W)
        acts  = self._acts.detach()           # (1, C, H, W)
        weights = grads.mean(dim=(2, 3), keepdim=True)
        cam   = (weights * acts).sum(dim=1).squeeze()
        cam   = torch.relu(cam).cpu().numpy()
        cam   = cv2.resize(cam, (320, 320))
        cam   = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, np.array(img.resize((320, 320)))

    def remove_hooks(self):
        self._fwd_hook.remove()
        self._bwd_hook.remove()


def overlay_cam(img_rgb: np.ndarray, cam: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    heatmap = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    return (img_rgb * (1 - alpha) + heatmap * alpha).astype(np.uint8)


# Use context that disables torch.no_grad so backward() works
grad_cam = GradCAM(model, model.backbone[28])
model.train()   # needed for gradients to flow; we won't update weights
print("GradCAM ready. Target: backbone[28] (last VGG16 conv layer)")

GradCAM ready. Target: backbone[28] (last VGG16 conv layer)


In [11]:
# ── Run Grad-CAM on the 6 worst failures ────────────────────────────────────
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
fig.suptitle(
    "Step 7 — Grad-CAM on Worst Failures  "
    "(Row 1: original  |  Row 2: ICM-A activation  |  Row 3: BE activation)",
    fontsize=11, fontweight="bold"
)

for col, (_, row) in enumerate(worst_failures.iterrows()):
    # ── Original image ────────────────────────────────────────────────────────
    img_arr = np.array(Image.open(row["path"]).convert("RGB").resize((320, 320)))
    axes[0, col].imshow(img_arr)
    axes[0, col].set_title(
        f"{row['image']}\nEUP={row['euploid_prob']:.2f}",
        fontsize=7, color="#e74c3c"
    )
    axes[0, col].axis("off")

    # ── Grad-CAM for ICM 'A' (class 2 = best ICM) ────────────────────────────
    try:
        cam_icm, img320 = grad_cam.generate(row["path"], target_head="ICM", target_class=2)
        axes[1, col].imshow(overlay_cam(img320, cam_icm))
        axes[1, col].set_title("ICM-A focus", fontsize=7)
    except Exception as e:
        axes[1, col].set_title(f"Error: {e}", fontsize=6)
    axes[1, col].axis("off")

    # ── Grad-CAM for BE 'Expanded' (class 2 = best expansion) ────────────────
    try:
        cam_be, img320 = grad_cam.generate(row["path"], target_head="BE", target_class=4)
        axes[2, col].imshow(overlay_cam(img320, cam_be))
        axes[2, col].set_title("BE-Expanded focus", fontsize=7)
    except Exception as e:
        axes[2, col].set_title(f"Error: {e}", fontsize=6)
    axes[2, col].axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "step7_gradcam_failures.png", dpi=120)
plt.show()
print("Saved → diag_output/step7_gradcam_failures.png")

# Restore eval mode after Grad-CAM
model.eval()
grad_cam.remove_hooks()
print("Model restored to eval mode, hooks removed.")

Saved → diag_output/step7_gradcam_failures.png
Model restored to eval mode, hooks removed.


---
## Step 8 — Heuristic Pre-Filter Evaluation

Test whether simple classical image-processing checks can catch arrested embryos **before** the model even runs. We test two independent signals:

1. **Fragmentation ratio** — ratio of small disconnected contours to total cellular area. High values = debris-heavy, likely arrested.
2. **Cavity ratio** — centre of the image vs outer ring brightness. A true blastocyst has a distinct blastocoel (fluid-filled cavity) that changes the inner brightness.

**Goal:** Find thresholds that catch ≥70% of arrested embryos while missing <10% of good ones.

In [12]:
def fragmentation_ratio(img_path: str) -> float:
    """
    Ratio of area from small contours (debris) to total foreground area.
    Higher = more fragmented.
    """
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.nan
    blur = cv2.GaussianBlur(img, (5, 5), 0)
    _, thresh = cv2.threshold(blur, 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    areas = [cv2.contourArea(c) for c in contours if cv2.contourArea(c) > 10]
    if not areas:
        return 0.0
    mean_area  = np.mean(areas)
    small_area = sum(a for a in areas if a < mean_area * 0.3)
    return small_area / (sum(areas) + 1e-8)


def cavity_ratio(img_path: str) -> float:
    """
    Ratio: mean brightness of centre 50% crop / full image.
    A blastocyst has a cavity → centre may be brighter (or distinctly different).
    Low ratio may indicate absence of organised internal structure.
    """
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.nan
    h, w = img.shape
    centre = img[h//4:3*h//4, w//4:3*w//4]
    return float(centre.mean()) / (float(img.mean()) + 1e-8)


print("Computing heuristic features on all test images...")
results["frag_ratio"]  = results["path"].apply(fragmentation_ratio)
results["cavity_ratio"] = results["path"].apply(cavity_ratio)

print(f"Done. Sample stats:")
for col in ["frag_ratio", "cavity_ratio"]:
    print(f"  {col}: mean={results[col].mean():.3f}, "
          f"std={results[col].std():.3f}, "
          f"min={results[col].min():.3f}, max={results[col].max():.3f}")

Computing heuristic features on all test images...
Done. Sample stats:
  frag_ratio: mean=0.032, std=0.013, min=0.000, max=0.070
  cavity_ratio: mean=0.975, std=0.044, min=0.848, max=1.082


In [13]:
# ── Sweep thresholds and find the best operating points ──────────────────────
from sklearn.metrics import roc_curve, auc

# Binary: is_arrested = EXP <= 1
results["is_arrested"] = (results["exp_gold"] <= 1).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Step 8 — Heuristic Filter Analysis", fontsize=13, fontweight="bold")

# ── Left: scatter frag_ratio vs euploid_prob ──────────────────────────────────
sc = axes[0].scatter(
    results["frag_ratio"], results["euploid_prob"],
    c=results["exp_gold"], cmap="RdYlGn", alpha=0.6, s=20
)
plt.colorbar(sc, ax=axes[0], label="EXP grade")
axes[0].set_xlabel("Fragmentation Ratio")
axes[0].set_ylabel("Model Euploid Prob")
axes[0].set_title("Frag Ratio vs Model Score")

# ── Middle: ROC curves for both heuristics at detecting arrested ──────────────
valid = results.dropna(subset=["frag_ratio", "cavity_ratio"])
for col, color, lbl in [
    ("frag_ratio",   "#e74c3c", "Frag ratio"),
    ("cavity_ratio", "#3498db", "Cavity ratio (inverted)"),
]:
    # For cavity_ratio, lower = more arrested, so invert it
    scores = valid[col] if col == "frag_ratio" else -valid[col]
    fpr, tpr, _ = roc_curve(valid["is_arrested"], scores)
    roc_auc = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, color=color, lw=2, label=f"{lbl} (AUC={roc_auc:.2f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("False Positive Rate (good embryos filtered out)")
axes[1].set_ylabel("True Positive Rate (arrested embryos caught)")
axes[1].set_title("ROC: Heuristics vs Arrested Detection")
axes[1].legend(fontsize=8)

# ── Right: box plots of heuristic features by EXP grade ──────────────────────
frag_by_grade = [results[results["exp_gold"] == g]["frag_ratio"].dropna().values
                 for g in grade_order]
bp2 = axes[2].boxplot(frag_by_grade, patch_artist=True)
for patch, color in zip(bp2["boxes"], box_colors):
    patch.set_facecolor(color)
axes[2].set_xticks(range(1, 6))
axes[2].set_xticklabels([f"EXP={g}" for g in grade_order], fontsize=9)
axes[2].set_ylabel("Fragmentation Ratio")
axes[2].set_title("Frag Ratio by Expansion Grade")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "step8_heuristic_filter.png", dpi=120)
plt.show()
print("Saved → diag_output/step8_heuristic_filter.png")

# ── Threshold sweep for fragmentation ratio ────────────────────────────────
print("\n── Fragmentation threshold sweep ───────────────────────────────────")
print(f"{'Threshold':>10} {'Arrested caught':>18} {'Good lost':>12} {'Net'}")
print("-" * 50)
for thresh in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]:
    flagged    = results[results["frag_ratio"] >= thresh]
    arr_caught = (flagged["is_arrested"] == 1).sum()
    good_lost  = (flagged["is_arrested"] == 0).sum()
    arr_total  = results["is_arrested"].sum()
    print(f"{thresh:>10.2f} {arr_caught:>5}/{arr_total:<12} {good_lost:>12}")

Saved → diag_output/step8_heuristic_filter.png

── Fragmentation threshold sweep ───────────────────────────────────
 Threshold    Arrested caught    Good lost Net
--------------------------------------------------
      0.10     0/54                      0
      0.15     0/54                      0
      0.20     0/54                      0
      0.25     0/54                      0
      0.30     0/54                      0
      0.35     0/54                      0


---
## Step 9 — Summary & Decision Matrix

Compile all findings from Steps 4–8 into a single diagnostic report.

In [14]:
print("═" * 65)
print("  EMBRYO AI DIAGNOSTIC REPORT")
print("═" * 65)

# ── Section 1: Dataset ────────────────────────────────────────────────────────
print("\n[1] DATASET")
print(f"    Gold test images processed : {len(results)}")
print(f"    Arrested embryos (EXP≤1)   : {len(arrested)}  "
      f"({len(arrested)/len(results)*100:.1f}%)")
print(f"    Normal embryos (EXP≥2)     : {len(normal)}  "
      f"({len(normal)/len(results)*100:.1f}%)")

# ── Section 2: Model accuracy ─────────────────────────────────────────────────
fp_count  = (arrested["predicted"] == True).sum()
fp_rate   = fp_count / len(arrested)
tn_count  = (normal["predicted"] == True).sum()
tp_rate   = tn_count / len(normal)

print("\n[2] MODEL ACCURACY")
print(f"    True positive rate (normal → euploid)   : {tp_rate*100:.1f}%")
print(f"    False positive rate (arrested → euploid): {fp_rate*100:.1f}%  "
      f"← {'🔴 CRITICAL' if fp_rate > 0.3 else '🟡 MODERATE' if fp_rate > 0.1 else '🟢 OK'}")

# ── Section 3: Head-level confusion ──────────────────────────────────────────
print("\n[3] HEAD-LEVEL CONFUSION (arrested group means)")
for metric, label in zip(metrics, metric_labels):
    arr_m = arrested[metric].mean()
    nor_m = normal[metric].mean()
    sep   = nor_m - arr_m
    status = "🔴 LOW" if sep < 0.15 else "🟡 MEDIUM" if sep < 0.25 else "🟢 GOOD"
    print(f"    {label:<20}: arr={arr_m:.3f} | nor={nor_m:.3f} | "
          f"sep={sep:.3f}  {status}")

# ── Section 4: Heuristic filter ───────────────────────────────────────────────
best_thresh  = 0.20
flagged_best = results[results["frag_ratio"] >= best_thresh]
arr_caught   = (flagged_best["is_arrested"] == 1).sum()
good_lost    = (flagged_best["is_arrested"] == 0).sum()

print("\n[4] HEURISTIC PRE-FILTER  (frag_ratio ≥ 0.20)")
print(f"    Arrested embryos caught : {arr_caught} / {len(arrested)} "
      f"({arr_caught/len(arrested)*100:.1f}%)")
print(f"    Good embryos falsely filtered: {good_lost} / {len(normal)} "
      f"({good_lost/len(normal)*100:.1f}%)")

# ── Section 5: Decision ───────────────────────────────────────────────────────
print("\n[5] RECOMMENDED ACTIONS  (based on findings above)")

if fp_rate > 0.5:
    print("    🔴 Decision A  → Implement heuristic filter NOW to stop false positives.")
    print("       Grad-CAM results will tell you if retraining is also needed.")
elif fp_rate > 0.2:
    print("    🟡 Decision A + B  → Add heuristic filter AND fine-tune on hard negatives.")
    print("       Review Grad-CAM output to confirm features are biologically plausible.")
else:
    print("    🟢 Decision B  → Fine-tune on hard negatives; basic features seem learnable.")

print()
print("    Grad-CAM review (manual):")
print("      → Check diag_output/step7_gradcam_failures.png")
print("      → If heatmap is on zona / background → retrain with clean labels (Decision C)")
print("      → If heatmap is on debris / inner region → fine-tune on hard negatives (Decision B)")
print()
print("    Output files generated:")
for f in sorted(OUTPUT_DIR.glob("*.png")) + sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"      {f.name}")

print("\n" + "═" * 65)

═════════════════════════════════════════════════════════════════
  EMBRYO AI DIAGNOSTIC REPORT
═════════════════════════════════════════════════════════════════

[1] DATASET
    Gold test images processed : 298
    Arrested embryos (EXP≤1)   : 54  (18.1%)
    Normal embryos (EXP≥2)     : 244  (81.9%)

[2] MODEL ACCURACY
    True positive rate (normal → euploid)   : 5.7%
    False positive rate (arrested → euploid): 72.2%  ← 🔴 CRITICAL

[3] HEAD-LEVEL CONFUSION (arrested group means)
    BE (Expansion)      : arr=0.493 | nor=0.616 | sep=0.123  🔴 LOW
    ICM Quality         : arr=0.580 | nor=0.217 | sep=-0.363  🔴 LOW
    TE Quality          : arr=0.467 | nor=0.275 | sep=-0.192  🔴 LOW
    Euploid Probability : arr=0.511 | nor=0.394 | sep=-0.117  🔴 LOW

[4] HEURISTIC PRE-FILTER  (frag_ratio ≥ 0.20)
    Arrested embryos caught : 0 / 54 (0.0%)
    Good embryos falsely filtered: 0 / 244 (0.0%)

[5] RECOMMENDED ACTIONS  (based on findings above)
    🔴 Decision A  → Implement heuristic filter 